# 02 · Gold source, notes cliniques et reconstruction

C'est ici que MedGemma (`google/medgemma-4b-it`) entre en jeu, à trois reprises :

1. **Gold source** : il trie les conditions candidates de chaque patient pour ne garder que les vraies conditions médicales → `data/gold_standard/`
2. **Notes** : il rédige une note clinique par patient, dans l'un de quatre styles → `data/processed/notes/`
3. **Reconstruction** : il relit chaque note et en extrait un dossier structuré → `data/processed/reconstruction/`

Il faut avoir lancé le notebook 01, et disposer d'un GPU (environ 1 h pour 40 patients sur 16 Go).

Vocabulaire utilisé dans le code :
- `gold` : le dossier de référence, celui qu'on cherche à retrouver ;
- `raw_note_text` / `note_text` : la note brute du modèle, et sa version nettoyée (la seule que voit la reconstruction) ;
- `recon_raw` / `recon_clean` / `recon_clipped` : le JSON brut, la version nettoyée (celle qu'on score), et l'intersection exacte avec le gold (pour déboguer).

In [ ]:
# Sur Google Colab seulement : récupérer le dépôt et installer les dépendances
# !git clone https://github.com/brettsamy-database/medical-data-reconstruction.git bupa
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Pour pouvoir importer `config` et `src`, que le notebook soit lancé depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

In [ ]:
# settings.N_PATIENTS = 5   # même valeur que dans le notebook 01

In [ ]:
# MedGemma est un modèle « gated » : il faut un token Hugging Face.
# En local, le mettre dans un fichier .env à la racine (HF_TOKEN=hf_..., voir .env.example) ;
# sur Colab, dans un secret nommé HF_TOKEN.
from src.llm import get_llm, hf_login

hf_login()
llm = get_llm()  # le modèle n'est chargé qu'au premier appel
print("Model:", settings.MODEL_NAME, "| transformers/torch available:", llm.available())

## 1. Retrouver l'échantillon du notebook 01

In [ ]:
from src import fhir

df_index = fhir.build_filtered_dataset()
df_sample, patients_prefilter, df_gold_prefilter = fhir.select_sample(df_index)
display(df_gold_prefilter)

## 2. Construire le gold source

Pour chaque patient, MedGemma reçoit la liste des conditions candidates et renvoie les index de celles qui sont
vraiment médicales. Il ne peut ni ajouter ni reformuler un élément, et en cas de doute on garde tout.
Les réponses sont mises en cache dans `data/processed/filter_cache/`.

In [ ]:
from src import gold

patients = gold.build_gold(patients_prefilter, llm=llm)
print("Gold dataset (after hybrid filter: prefilter then LLM)")
display(gold.gold_dataframe(patients))

## 3. Générer les notes

Les styles sont attribués à tour de rôle (10 patients par style sur 40) et enregistrés dans le gold, pour pouvoir
comparer les résultats par style. La génération est la seule étape aléatoire du pipeline (température 0.8).

In [ ]:
from src import notes

patients = notes.assign_styles_and_save(patients)  # le style fait partie du gold
df_gold = gold.gold_dataframe(patients)
print("Note style distribution")
display(df_gold["note_style"].value_counts().to_frame("n"))

In [ ]:
raw_notes, generated_notes = notes.generate_notes(patients, llm=llm)
df_notes = notes.notes_dataframe(patients, raw_notes, generated_notes)
print("Notes dataset (note_text = cleaned note used for reconstruction)")
display(df_notes)

## 4. Reconstruire le dossier à partir de la note

Le modèle ne voit que la note nettoyée. Il renvoie un JSON (nom, date de naissance, genre, conditions,
prescriptions), qu'on nettoie ensuite : puces, doublons, éléments trop longs, bruit social.

In [ ]:
from src import reconstruction

recon_raw, recon_clean, recon_clipped = reconstruction.reconstruct_all(patients, generated_notes, llm=llm)
df_recon = reconstruction.recon_dataframe(recon_raw, recon_clean, recon_clipped)
print("Reconstruction dataset (main = cleaned/no strict clipping, clipped = exact-overlap debug only)")
display(df_recon)

**Suite :** `03_benchmark_results.ipynb`. Tout est relu depuis le disque, on peut donc le lancer dans une autre session.